# 060 — Round 4: grouped k-fold training

`fixing.md` #4. A genuine held-out **variance** estimate — how much the metrics move
across independent train/val splits — instead of the single fixed split every other
notebook uses.

**Locked model** (decided from `C4`/`C5`, `final-comments.md`): `attention_unet_nll`
— top-2 on every detection cut, NLL head covers both `structural delta` (from `μ`)
and `structural z`. Its 3 folds are **already trained** (see §4). `k = settings.KFOLD_K`
(3), `β = settings.NLL_BETA` (0.5), fixed.

**Any other trained model can now be k-folded too.** `MODEL_SPECS` (§1) lists every
architecture trained anywhere in the project; `SELECTED` is the shortlist to run this
session — **uncomment one model at a time** and run. Each `(model, fold)` is one
`train_single.py` subprocess (fresh GPU context, `fixing.md` §7.1); the loop is
**skip-if-exists**, so re-running only trains what's missing. `MAX_MODELS_PER_RUN` /
`MAX_FOLDS_PER_RUN` cap the work per session.

**Split** (`scripts.kfold`, model-independent): real artworks partitioned into `k`
groups *by artwork ID* (no section leakage); fold `i` holds out group `i` as
validation, the rest **plus all mockups** are train. No test split — the held-out
artworks are the fold's evaluation set. `data/test/` (GT paintings) is untouched.

Realistic per fold on this hardware: ~1–3 h (NLL runs early-stop ~epoch 25; the
100-epoch cap is the upper bound). Evaluate with `061_kfold_evaluation.ipynb` once
≥ 2 folds of a model exist.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [1]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports.

In [2]:
import json
import subprocess

from scripts.config import settings
from scripts.dataset import load_image_pairs
from scripts.kfold import fold_artwork_groups, grouped_kfold_splits

print(f"KFOLD_K = {settings.KFOLD_K}   KFOLD_SEED = {settings.KFOLD_SEED}   NLL_BETA = {settings.NLL_BETA}")

KFOLD_K = 3   KFOLD_SEED = 42   NLL_BETA = 0.5


## 1. Model registry, selection, and the fold plan

`MODEL_SPECS` — every architecture trained in the project, with exactly the flags
`train_single.py` needs (NLL vs deterministic, builder kwargs, two-phase warm-start).
`SELECTED` — what to k-fold this run: **uncomment one at a time**.

The fold plan itself is model-independent: deterministic in `(KFOLD_K, KFOLD_SEED)`
and the set of real-artwork IDs. Mockups are in every fold's train set.

In [3]:
K = settings.KFOLD_K
KFOLD_DIR = settings.MODELS_DIR / "kfold"
KFOLD_LOG_DIR = settings.LOGS_DIR / "kfold"

# Every model trained anywhere in the project. Fields (all optional except when noted):
#   nll        -> True adds  --nll --loss-name laplace_nll --nll-beta <settings.NLL_BETA>
#   kwargs     -> JSON builder kwargs passed to train_single --kwargs
#   epochs     -> defaults to settings.EPOCHS
#   lr         -> defaults to settings.LEARNING_RATE (the _ft phase-2 runs override it)
#   init_from  -> name of another model in this SAME fold dir to warm-start weights
#                 from (fixing.md #6 two-phase fine-tuning). That model must be
#                 trained for the fold first — put it earlier in SELECTED.
MODEL_SPECS = {
    # ---- deterministic (signal: `structural delta` only — no sigma / structural z) ----
    "unet":                 dict(nll=False, kwargs={}),
    "resunet":              dict(nll=False, kwargs={}),
    "attention_unet":       dict(nll=False, kwargs={}),
    "unet_v2":              dict(nll=False, kwargs=dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2)),
    "unet_restormer":       dict(nll=False, kwargs=dict(num_heads=8, ffn_expansion_factor=2)),
    "unet_dilated":         dict(nll=False, kwargs={}),
    "unet_v2_dilated":      dict(nll=False, kwargs=dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2)),
    "efficientnet_unet":    dict(nll=False, kwargs={}),
    "efficientnet_unet_ft": dict(nll=False, kwargs={"freeze_encoder": False},
                                 init_from="efficientnet_unet",
                                 epochs=settings.FINETUNE_EPOCHS, lr=settings.FINETUNE_LEARNING_RATE),
    # ---- NLL / heteroscedastic (signals: `structural delta` from mu + `structural z` from mu/sigma) ----
    "unet_nll":                 dict(nll=True, kwargs={}),
    "resunet_nll":              dict(nll=True, kwargs={}),
    "attention_unet_nll":       dict(nll=True, kwargs={}),
    "efficientnet_unet_nll":    dict(nll=True, kwargs={}),
    "efficientnet_unet_nll_ft": dict(nll=True, kwargs={"freeze_encoder": False},
                                     init_from="efficientnet_unet_nll",
                                     epochs=settings.FINETUNE_EPOCHS, lr=settings.FINETUNE_LEARNING_RATE),
}

# Models to k-fold this session. Uncomment ONE at a time and run §2.
# `attention_unet_nll` is the locked Round 4 model (folds 0-2 already trained -> skipped).
SELECTED = [
    #"attention_unet_nll",
    # "unet_nll",
    "resunet_nll",
    # "efficientnet_unet_nll",
    # "unet",
    # "resunet",
    # "attention_unet",
    # "unet_v2",
    # "unet_restormer",
    # "unet_dilated",
    # "unet_v2_dilated",
    # "efficientnet_unet",
    # "efficientnet_unet_ft",         # select "efficientnet_unet" together with this, in this order
    # "efficientnet_unet_nll_ft",     # select "efficientnet_unet_nll" together with this, in this order
]

DEFAULTS = dict(nll=False, kwargs={}, loss_name="laplace_nll", epochs=settings.EPOCHS,
                lr=None, init_from=None)


def spec(name):
    s = dict(DEFAULTS)
    s.update(MODEL_SPECS[name])
    return s


pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
held_out = fold_artwork_groups(pairs, k=K, seed=settings.KFOLD_SEED)
splits = grouped_kfold_splits(pairs, k=K, seed=settings.KFOLD_SEED)

print(f"K = {K}   KFOLD_SEED = {settings.KFOLD_SEED}   NLL_BETA = {settings.NLL_BETA}")
print(f"SELECTED this run: {SELECTED}\n")
print(f"{len(pairs)} pairs total\n")
for i, (groups, (tr, va)) in enumerate(zip(held_out, splits)):
    print(f"fold {i}: held out {len(groups)} artworks -> val {len(va)} pairs / train {len(tr)} pairs")
    print(f"         {groups}")

K = 3   KFOLD_SEED = 42   NLL_BETA = 0.5
SELECTED this run: ['resunet_nll']

1164 pairs total

fold 0: held out 9 artworks -> val 333 pairs / train 831 pairs
         ['mano', 'mod', 'natmorta2', 'q2', 'q3', 'santo', 'sch02', 'sch03', 'volto']
fold 1: held out 8 artworks -> val 221 pairs / train 943 pairs
         ['a1', 'c1', 'cristo', 'natmorta3', 'oblato_tot', 'oblato_volto', 'sch01', 'testa']
fold 2: held out 8 artworks -> val 214 pairs / train 950 pairs
         ['a2', 'b1', 'corpo', 'natmorta1', 'orecchio', 'q1', 'torso', 'veste']


## 2. Train — one subprocess per `(model, fold)`, resumable

For each model in `SELECTED`, `scripts.train_single --fold i --kfold-k K` builds
fold `i`'s split itself (deterministic from `settings`). Checkpoints land in
`models/kfold/fold_<i>/<model>/best_model.keras` (+ `history.json`).

A `(model, fold)` whose checkpoint already exists is skipped. `MAX_MODELS_PER_RUN`
caps how many models this run trains (each = up to `K` folds); `MAX_FOLDS_PER_RUN`
caps folds per model per run. Set both to `1` for a single fold per session.

In [4]:
MAX_MODELS_PER_RUN = 1  # e.g. 1 to train a single model per session
MAX_FOLDS_PER_RUN = 3   # e.g. 1 to train a single fold (per model) per session


def build_cmd(name, fold, model_dir):
    s = spec(name)
    cmd = [
        sys.executable, "-m", "scripts.train_single",
        "--arch", name,
        "--epochs", str(s["epochs"]),
        "--model-dir", str(model_dir),
        "--log-dir", str(KFOLD_LOG_DIR / f"fold_{fold}"),
        "--kwargs", json.dumps(s["kwargs"]),
        "--fold", str(fold),
        "--kfold-k", str(K),
    ]
    if s["nll"]:
        cmd += ["--nll", "--loss-name", s["loss_name"], "--nll-beta", str(settings.NLL_BETA)]
    if s["lr"] is not None:
        cmd += ["--lr", str(s["lr"])]
    if s["init_from"] is not None:
        src = model_dir / s["init_from"] / "best_model.keras"
        if not src.exists():
            raise FileNotFoundError(
                f"{name} fold {fold} warm-starts from {src}, which is missing. "
                f"Add '{s['init_from']}' to SELECTED before '{name}' and re-run."
            )
        cmd += ["--init-from", str(src)]
    return cmd


models_this_run = 0
for name in SELECTED:
    if MAX_MODELS_PER_RUN is not None and models_this_run >= MAX_MODELS_PER_RUN:
        print(f"[stop] MAX_MODELS_PER_RUN={MAX_MODELS_PER_RUN} reached — re-run to continue")
        break
    folds_this_run = 0
    trained_any = False
    for fold in range(K):
        model_dir = KFOLD_DIR / f"fold_{fold}"
        ckpt = model_dir / name / "best_model.keras"
        if ckpt.exists():
            print(f"[skip] {name} fold {fold} — checkpoint at {ckpt}")
            continue
        if MAX_FOLDS_PER_RUN is not None and folds_this_run >= MAX_FOLDS_PER_RUN:
            print(f"[stop] {name}: MAX_FOLDS_PER_RUN={MAX_FOLDS_PER_RUN} reached")
            break

        print(f"\n{'=' * 60}\n  {name} — fold {fold}/{K}\n{'=' * 60}")
        subprocess.run(build_cmd(name, fold, model_dir), cwd=project_root, check=True)
        folds_this_run += 1
        trained_any = True

        hist = json.loads((model_dir / name / "history.json").read_text())
        print(f"\n{name} fold {fold}: best val_loss = {min(hist['val_loss']):.4f}  "
              f"({len(hist['val_loss'])} epochs)")

    if trained_any:
        models_this_run += 1

print(f"\ntrained folds for {models_this_run} model(s) this run")


  resunet_nll — fold 0/3
k-fold: fold 0/3  train=831  val(held-out)=333


2026-08-27 23:20:19.655028: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-08-27 23:20:19.655054: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-08-27 23:20:19.655060: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-08-27 23:20:19.655265: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-27 23:20:19.655277: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



  Architecture: resunet_nll (laplace_nll, beta=0.5)
Model: "resunet_nll"
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_layer           │ (None, None,      │           0 │ -                  │
│ (InputLayer)          │ None, 3)          │             │                    │
├───────────────────────┼──────��────────────┼─────────────┼────────────────────┤
│ conv2d_1 (Conv2D)     │ (None, None,      │       1,728 │ input_layer[0][0]  │
│                       │ None, 64)         │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ group_normalization_1 │ (None, None,      │         128 │ conv2d_1[0][0]     │
│ (GroupNormalization)  │ None, 64)         │             │                    │
├───────────────────────┼─────────

/opt/miniconda3/envs/deep-layers/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-08-27 23:20:22.319203: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.2190 - mae: 0.1873 - psnr: 14.2254 - ssim: 0.1374
Epoch 1: val_loss improved from None to -0.30075, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_0/resunet_nll/best_model.keras

Epoch 1: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_0/resunet_nll/best_model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 154s 1s/step - loss: -0.2190 - mae: 0.1873 - psnr: 14.2254 - ssim: 0.1374 - val_loss: -0.3007 - val_mae: 0.1567 - val_psnr: 15.4978 - val_ssim: 0.2040 - learning_rate: 1.0000e-04
Epoch 2/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.3489 - mae: 0.1415 - psnr: 16.6946 - ssim: 0.2486
Epoch 2: val_loss improved from -0.30075 to -0.33251, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_0/resunet_nll/best_model.keras

Epoch 2: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_0/resunet_n

...

118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.2316 - mae: 0.1805 - psnr: 14.5694 - ssim: 0.1548
Epoch 1: val_loss improved from None to -0.28831, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_1/resunet_nll/best_model.keras

Epoch 1: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_1/resunet_nll/best_model.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 170s 1s/step - loss: -0.2316 - mae: 0.1805 - psnr: 14.5694 - ssim: 0.1548 - val_loss: -0.2883 - val_mae: 0.1658 - val_psnr: 15.7613 - val_ssim: 0.1883 - learning_rate: 1.0000e-04
Epoch 2/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.3511 - mae: 0.1407 - psnr: 16.9063 - ssim: 0.2520
Epoch 2: val_loss improved from -0.28831 to -0.39585, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_1/resunet_nll/best_model.keras

Epoch 2: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_1/resunet_n

2026-08-28 02:07:36.814272: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-08-28 02:07:36.814291: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-08-28 02:07:36.814297: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-08-28 02:07:36.814308: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-28 02:07:36.814318: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



  Architecture: resunet_nll (laplace_nll, beta=0.5)
Model: "resunet_nll"
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_layer           │ (None, None,      │           0 │ -                  │
│ (InputLayer)          │ None, 3)          │             │                    │
├───────────────────────┼──────��────────────┼─────────────┼────────────────────┤
│ conv2d_1 (Conv2D)     │ (None, None,      │       1,728 │ input_layer[0][0]  │
│                       │ None, 64)         │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ group_normalization_1 │ (None, None,      │         128 │ conv2d_1[0][0]     │
│ (GroupNormalization)  │ None, 64)         │             │                    │
├───────────────────────┼─────────

/opt/miniconda3/envs/deep-layers/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-08-28 02:07:39.584369: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.2352 - mae: 0.1812 - psnr: 14.6362 - ssim: 0.1526
Epoch 1: val_loss improved from None to -0.33759, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/resunet_nll/best_model.keras

Epoch 1: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/resunet_nll/best_model.keras
119/119 ━━━━━━━━━━━━━━━━━━━━ 172s 1s/step - loss: -0.2352 - mae: 0.1812 - psnr: 14.6362 - ssim: 0.1526 - val_loss: -0.3376 - val_mae: 0.1523 - val_psnr: 15.9260 - val_ssim: 0.2153 - learning_rate: 1.0000e-04
Epoch 2/100
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.3508 - mae: 0.1398 - psnr: 17.0218 - ssim: 0.2615
Epoch 2: val_loss improved from -0.33759 to -0.38678, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/resunet_nll/best_model.keras

Epoch 2: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/resunet_n

## 3. Status

In [5]:
on_disk = sorted({
    p.parent.name
    for f in range(K)
    for p in (KFOLD_DIR / f"fold_{f}").glob("*/best_model.keras")
} | set(SELECTED))

ready_for_eval = []
for name in on_disk:
    print(f"\n{name}")
    print(f"  {'fold':<6}{'checkpoint':<12}{'epochs':<9}{'best val_loss'}")
    print("  " + "-" * 42)
    n_done = 0
    for fold in range(K):
        hp = KFOLD_DIR / f"fold_{fold}" / name / "history.json"
        ck = (KFOLD_DIR / f"fold_{fold}" / name / "best_model.keras").exists()
        n_done += ck
        if hp.exists():
            h = json.loads(hp.read_text())
            print(f"  {fold:<6}{'ok' if ck else 'MISSING':<12}{len(h['val_loss']):<9}{min(h['val_loss']):.4f}")
        else:
            print(f"  {fold:<6}{'-':<12}{'-':<9}-")
    if n_done >= 2:
        ready_for_eval.append(name)

print(f"\nmodels with >= 2 folds (evaluate in 061): {ready_for_eval or 'none yet'}")


attention_unet_nll
  fold  checkpoint  epochs   best val_loss
  ------------------------------------------
  0     ok          22       -0.3844
  1     ok          38       -0.4451
  2     ok          24       -0.4424

resunet_nll
  fold  checkpoint  epochs   best val_loss
  ------------------------------------------
  0     ok          36       -0.3811
  1     ok          30       -0.4583
  2     ok          34       -0.4465

models with >= 2 folds (evaluate in 061): ['attention_unet_nll', 'resunet_nll']


## 4. Result — `attention_unet_nll`, all 3 folds trained (2026-08-27)

The locked Round 4 model. Ran across a single session (each fold early-stopped well
before the 100-epoch cap). Fold plan is deterministic in `(KFOLD_K=3, KFOLD_SEED=42)`:

| fold | held-out artworks | val / train pairs | epochs | best `val_loss` |
|---|---|---|---|---|
| 0 | `mano` `mod` `natmorta2` `q2` `q3` `santo` `sch02` `sch03` `volto` | 333 / 831 | 22 | −0.3844 |
| 1 | `a1` `c1` `cristo` `natmorta3` `oblato_tot` `oblato_volto` `sch01` `testa` | 221 / 943 | 38 | −0.4451 |
| 2 | `a2` `b1` `corpo` `natmorta1` `orecchio` `q1` `torso` `veste` | 214 / 950 | 24 | −0.4424 |

No NaN, no instability — the three §7 training-bug fixes hold on the k-fold splits
too. Fold 0 holds out the largest / hardest group of artworks and lands ~0.06 higher
in `val_loss` than folds 1–2, which agree tightly.

Checkpoints in `models/kfold/fold_<i>/attention_unet_nll/best_model.keras`
(filesystem only — `models/` is gitignored). Evaluation and the finding-#4 verdict
are in `061_kfold_evaluation.ipynb` (§6): fold-to-fold AUROC std ≈ 0.008, ensemble
`structural z` AUROC 0.72 — the single-split metrics used elsewhere in the project
are confirmed trustworthy.

**Other models:** none k-folded yet. Add one to `SELECTED` (§1) and re-run §2/§3 to
extend the variance check — `061` picks up any model with ≥ 2 folds on disk.